In [3]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import xarray as xr
from glob import glob
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter, LatitudeLocator
import cartopy.feature as cfeature
import config.STATIC as call


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'config'

In [ ]:
iY,iX=40,30

In [ ]:
fd=xr.open_dataset('fd_USDM_classification_0.125_degrees.nc')



In [ ]:
def plot_percent_of_year_FD(fd):
    time_start = '2002-01-01'
    time_end = '2021-12-31'
    
    cmap = plt.get_cmap('YlOrRd')

    # bad_color = cmap.get_bad()
    # print(bad_color)
    
    save_dir = f'{call.fig_dir}/USDM_figs'
    os.system(f'mkdir -p {save_dir}')

    yearly_sum = fd.resample(time='1Y').sum().clip(max=1).mean(dim='time')
    
    data_arrays = [yearly_sum['fd_start']]
    
    lon = yr_plot1.lon.values
    lat = yr_plot1.lat.values

    fig, axs = plt.subplots(
    nrows = 3, ncols= 3, subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(15, 9),dpi=300)
    
    axs = axs.flatten()

    axs_start = 0
    for idx, (arr, var_name) in enumerate(zip(data_arrays,['Percent of year with\nFlash Drought'])):

        data = arr.mean(dim='time').values
        data = np.where(np.isnan(land_mask),np.nan,data)
        # data = np.ma.masked_invalid(data)
        for Y in range(data.shape[0]):
            for X in range(data.shape[1]):
                if np.isnan(land_mask[Y,X]) or (data[Y,X] == 0):
                    data[Y,X]=np.nan

        # print(data)
        if 'diff' in var_name:
            v = np.linspace(np.nanmin(data), np.nanmax(data), 9, endpoint=True)
            v = [i for i in v if i < 0] + [0] + [i for i in v if i > 0]
            norm = mcolors.TwoSlopeNorm(vmin=v[0], vcenter=0, vmax=v[-1])
        else:
            v = np.linspace(np.nanmin(data), np.nanmax(data), 8, endpoint=True)

            
        map = Basemap(projection='cyl', llcrnrlat=25, urcrnrlat=50,
              llcrnrlon=-128, urcrnrlon=-60, resolution='l')
        x, y = map(*np.meshgrid(lon, lat))

        if 'diff' in var_name:
            im = axs[axs_start].contourf(x, y, data, levels=v, extend='both',
                          transform=ccrs.PlateCarree(), cmap=cmap_diff,norm=norm)
        else:
            im = axs[axs_start].contourf(x, y, data, levels=v, extend='both',
                          transform=ccrs.PlateCarree(), cmap=cmap)
            
        gl = axs[axs_start].gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                                   linewidth=0.7, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.left_labels = True
        gl.bottom_labels = True
        gl.xformatter = LongitudeFormatter()
        gl.yformatter = LatitudeFormatter()
        axs[axs_start].coastlines()
        axs[axs_start].set_aspect('equal')  # this makes the plots better
        if axs_start <=2:
            axs[axs_start].set_title(f'{var_name}: {year_ranges_tuple_1[0]}-{year_ranges_tuple_1[1]}',fontsize=12)
        elif axs_start in [3,4,5]:
            axs[axs_start].set_title(f'{var_name}: {year_ranges_tuple_2[0]}-{year_ranges_tuple_2[1]}',fontsize=12)
        else:
            axs[axs_start].set_title(f'{var_name}\n[bottom - top]',fontsize=12)
        cbar = fig.colorbar(im, ax = axs[axs_start], orientation='horizontal')
        cbar.set_label('mm/year', fontsize=10, labelpad=5)
        axs_start+=1

    plt.suptitle(f'Accumulated yearly average of variable.\nThen averaged over the period. ',fontsize=20)
    plt.tight_layout()
    plt.savefig(f'{save_dir}/yearly_EVP_PET_refET_sum.png')

    return('Completed yearly sum plots')

In [ ]:
yearly_sum = fd.resample(time='1Y').sum().clip(max=1).mean(dim='time')
yearly_sum.fd_start.plot()

In [ ]:
a.fd_start.sel(time='2012-05-20',method='nearest').plot()

In [ ]:



def step_1_find_if_2_category_change(usdm):
    '''Find if a.) 2 category change in 2 weeks, b.) that the category is minimially sustained at that threshold'''
    print('Working on finding if there is a 2 category change over 2 weeks and sustained for 2 weeks')
    file_out = usdm.copy(deep=True).rename({'usdm':'fd_start'})
    file_out.fd_start[:,:,:] = 0

    for iDate, date in enumerate(usdm.time.values):
        if (iDate >=4):
            #check if there is a +2 category decrease (increase in this dataset) in 2 weeks and sustained for another 2 weeks
            prev_4wk = usdm.usdm[iDate-4,:,:].values
            prev_2wk = usdm.usdm[iDate-2,:,:].values
            prev_2wks = usdm.usdm[iDate-1:iDate+1,:,:].values
            current_wk = usdm.usdm[iDate,:,:].values
            
            # Find if 2 category change
            cat_change = np.where((prev_2wk - prev_4wk >=2),1,0)

            #Find if iDate-1 and iDate are >= prev_2wk
            sustained_change = np.all((prev_2wks >= prev_2wk),axis=0)

            fd_classification = np.where((cat_change==1) & (sustained_change==1),1,0)
            
            file_out.fd_start[iDate,:,:] = fd_classification
    return file_out


def continue_FD_if_category_is_not_equal_to_zero(usdm,fd_step1):
    ''' if a.) current usdm category is greater than or equal to 0;
    and b.) current FD classification is 0, but last weeks was 1'''

    '''This code should technically find the end of flash drought too, 
    since when the category returns to 0 then the flash drough has ended'''

    print('Working on finding weeks when flash drought continued and eventually returns back to category of zero.')
    fd_out = fd_step1.copy(deep=True)
    fd_out['fd_2'] = fd_out['fd_start'].copy(deep=True)

    usdm_arr = usdm.usdm.values
    step_1_arr_vals = fd_out['fd_start'].values
    fill_array = step_1_arr_vals.copy()

    for iY, Y in enumerate(fd_out.lat.values):
        print(f'Working on lat idx {iY} out of {len(fd_out.lat.values)}.')
        for iX, X in enumerate(fd_out.lon.values):
            
            for iDate, date in enumerate(fd_out.time.values):
                if (iDate >=4):

                    #if usdm current week is greater than 1
                    if usdm_arr[iDate,iY,iX] >= 1:
                        #if FD is presnet in previous week
                        if (step_1_arr_vals[iDate,iY,iX] == 0) & (step_1_arr_vals[iDate-1,iY,iX] == 1):
                            fill_array[iDate,iY,iX] = 1
    fd_out['fd_2'][:,:,:] = fill_array
    
    return fd_out


# def end_flash_drought(usdm,fd_step1, fd_step2):
#     fd_out = fd_step2.copy(deep=True)
#     fd_out['fd_3'] = fd_out['fd_2'].copy(deep=True)

#     for iY, Y in enumerate(fd_out.lat.values):
#         for iX, X in enumerate(fd_out.lon.values):
            
#             for iDate, date in enumerate(fd_out.time.values):
#                 if (iDate >=4) and (iDate < len(fd_out.time.values)-4):

#                     #if usdm current week is greater than previous week
#                     if usdm['usdm'][iDate,iY,iX].values >= usdm['usdm'][iDate-1,iY,iX].values:
#                         #if FD is presnet in previous week
#                         if (fd_step1['fd_1'][iDate,iY,iX].values == 0) & (fd_step1['fd_1'][iDate-1,iY,iX].values == 1):
#                             fd_out['fd_2'][iDate,iY,iX] = 1

def usdm_flash_drought():
    fd_start = step_1_find_if_2_category_change(usdm)
    fd_continue = continue_FD_if_category_is_not_equal_to_zero(usdm,fd_start)
    fd_continue.to_netcdf('fd_USDM_classification_0.125_degrees.nc')
    '''after this step, remap with cdo using nearest neigbor'''
usdm_flash_drought()     

In [ ]:
fd_step1.fd_1[657,:,:].plot()

In [ ]:
fd = xr.open_dataset(f'/glade/work/klesinger/sesr/Data/USDM/fd_USDM_classification_0.125_degrees.nc')
fd

# Plot annual frequency

In [ ]:
fd.fd_2.sel(time='2024-05-09',method='nearest').plot()

In [ ]:
fd = xr.open_dataset(f'/glade/work/klesinger/sesr/Data/USDM/fd_USDM_classification_0.50_degrees.nc')

state_boundaries = cfeature.NaturalEarthFeature(
    category='cultural',
    name='admin_1_states_provinces_lines',
    scale='50m',
    facecolor='none',
    edgecolor='black'  # You can change the color of the state boundaries here
)


lat_idx = 23
lon_idx = 66

lat = fd.lat.values[lat_idx]
lon = fd.lon.values[lon_idx]

#plot location 
location = fd.fd_2
loc_arr = location.values
loc_arr[:,:,:] = np.nan
loc_arr[:,lat_idx,lon_idx] = 1

location[:,:,:] = loc_arr


# location[300,:,:].plot()

# Plotting a specific slice (time index 300)
fig = plt.figure(figsize=(10, 6))
ax = plt.subplot(1, 1, 1, projection=ccrs.PlateCarree())

# Plot data slice
location[300, :, :].plot(ax=ax, transform=ccrs.PlateCarree(), cmap='Reds', vmin=0, vmax=2)
# Plot the bright star at the specific latitude and longitude
ax.plot(lon, lat, marker='*', color='yellow', markersize=15, markeredgecolor='black', transform=ccrs.PlateCarree())

# Add map features
ax.add_feature(cfeature.BORDERS, linestyle='-', linewidth=1)
ax.add_feature(cfeature.COASTLINE, linewidth=1)
ax.add_feature(cfeature.RIVERS, linestyle='-', linewidth=0.5, edgecolor='blue')
ax.add_feature(cfeature.LAKES, alpha=0.5, edgecolor='blue')
ax.add_feature(cfeature.LAND, facecolor='none')
ax.add_feature(cfeature.OCEAN)
# ax.add_feature(cfeature.MOUNTAINS, facecolor='gray', alpha=0.3)  # Adding mountains feature
# Add state boundary lines
ax.add_feature(state_boundaries, linestyle='-', linewidth=0.5)  # Adjust linewidth as needed


# Set plot title and labels
ax.set_title('Data at time index 300 with US Border', fontsize=14)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)